# 03 - Model vs NWS flood warnings (K-fold CV)

Benchmarks the trained GOES/GLM model against **NWS Flash-Flood + Areal-Flood WARNINGS** -
the operational forecaster baseline. Both are rasterized to the 50 km grid on the **fixed
test set** (the held-out CV test months, spanning all of 2019-2025) and scored against the
**observed-flood labels** (Groundsource union NCEI storm events - the same `y` the model was
trained on).

The model prediction is the **K-fold ensemble**: each model is trained once per CV fold
(`foldsplit.py`), and its test prediction is the mean of the fold-models' probabilities on
the fixed test set. The operating threshold is the best-F1 point on the **pooled CV
validation** (each fold's own val, which together partition the non-test pool).

Note on fairness: NWS warnings are same-day, short-lead nowcasts, while our model predicts
day D from **D-1** GOES (a ~1-day lead). So this is "who better flags the cells that flood",
not a like-for-like lead comparison - the model is doing a strictly harder (longer-lead) task.

Sections: (1) model predictions, (2) warnings -> 50 km grid, (3) metrics table
(model & warnings vs observed), (3b) single models vs ensembles, (4) who catches the observed
floods, (5) per-day maps, (6) aggregate spatial view, (7) metric bar chart.


## 0. Setup

In [ ]:
import os
import pickle
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve
from torch.utils.data import DataLoader

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
MODEL_DIR = ROOT / "notebooks" / "model"
OUT_DIR = MODEL_DIR / "outputs"
sys.path.insert(0, str(MODEL_DIR))
sys.path.insert(0, str(MODEL_DIR / "trainers"))

from config import STATES_GEOJSON, UNIFIED_PARQUET, build_grid_cells
from gridindex import build_pix2cell
import foldsplit
import resnet3d, cnn_attn, convgru_attn   # noqa: E401  (deep)
import xgb                                  # noqa: E401  (tabular trainers)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DEEP = {"resnet3d": resnet3d, "cnn_attn": cnn_attn, "convgru_attn": convgru_attn}
TAB = {"xgb": xgb}
N_FOLDS = foldsplit.N_FOLDS

_, GRID_R, GRID_C, land = build_pix2cell()
CACHE_DIR = resnet3d.CACHE_DIR


def fold_days(k):
    """Set the fold and return (train, val, test); test is fixed across folds."""
    os.environ["FOLD"] = str(k)
    return foldsplit.fold_splits(CACHE_DIR)


_, _, te_days = fold_days(0)                    # fixed held-out test set


def base_rate(days):
    y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days])
    return float(y[:, land].mean())


print(f"device {DEVICE} | grid {GRID_R}x{GRID_C} | land {int(land.sum())} cells | "
      f"{N_FOLDS}-fold CV")
print(f"fixed test: {len(te_days)} days  base rate {base_rate(te_days):.4f}")

# grid polygons + state outlines for maps + rasterizing warnings
cells_gdf, _, _, _ = build_grid_cells()
cells_albers = cells_gdf.to_crs(5070)
RR, CC = cells_albers["R"].to_numpy(), cells_albers["C"].to_numpy()
states = gpd.read_file(STATES_GEOJSON)
states = states[~states["name"].isin(["Alaska", "Hawaii", "Puerto Rico"])].to_crs(5070)


## 1. Model predictions (K-fold ensemble)

The model's test prediction is the **mean over CV folds** of each fold-model's probability on
the fixed test set. The operating threshold is the best-F1 point on the **pooled CV
validation** (every fold scored on its own val, concatenated). `MODEL` may be a single model
name or a `{name: weight}` dict to blend several models' fold-ensembles.


In [ ]:
# a single model name, or a dict {name: weight} to ENSEMBLE (weighted-avg probability).
# Each model is itself a K-fold ensemble over its per-fold checkpoints.
MODEL = "convgru_attn"        # best single model (see 3b): beats every ensemble on F1/CSI
# examples:  {"resnet3d": 0.5, "convgru_attn": 0.5}  |  {"resnet3d": 0.4, "convgru_attn": 0.4, "xgb": 0.2}


def _fold_ckpt(name, k):
    ext = "pt" if name in DEEP else "pkl"
    return OUT_DIR / f"{name}_f{k}.{ext}"


def _folds_avail(name):
    return [k for k in range(N_FOLDS) if _fold_ckpt(name, k).exists()]


@torch.no_grad()
def _infer_ckpt(name, ckpt, days):
    """(probs, trues) for one fold's checkpoint of one model."""
    if name in DEEP:
        mod = DEEP[name]
        net = mod.FloodNet().to(DEVICE)                   # stats are saved buffers
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        net.eval()
        probs, trues = [], []
        loader = DataLoader(mod.FeatureCache(days), batch_size=16, num_workers=8)
        for seq, summ, t, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                pr = torch.sigmoid(net(seq.to(DEVICE).float(), summ.to(DEVICE).float(),
                                       t.to(DEVICE).float())).squeeze(1)
            probs.append(pr.float().cpu().numpy())
            trues.append(y.numpy())
        del net
        torch.cuda.empty_cache()
        return np.concatenate(probs), np.concatenate(trues)
    payload = pickle.load(open(ckpt, "rb"))               # tabular
    return TAB[name].predict_grids(payload, days, land)


_CVCACHE = {}


def cv_model(name):
    """Per-model CV: fixed-test ensemble probs + pooled-val probs/trues (folds concat)."""
    if name not in _CVCACHE:
        tprobs, vpr, vtr = [], [], []
        for k in _folds_avail(name):
            _, va_k, _ = fold_days(k)
            vp, vy = _infer_ckpt(name, _fold_ckpt(name, k), va_k)   # fold-k own val
            tp, _ = _infer_ckpt(name, _fold_ckpt(name, k), te_days)  # fixed test
            vpr.append(vp); vtr.append(vy); tprobs.append(tp)
        _CVCACHE[name] = dict(test=np.mean(tprobs, axis=0),
                              vprobs=np.concatenate(vpr),
                              vtrues=np.concatenate(vtr))
    return _CVCACHE[name]


def spec_name(spec):
    if isinstance(spec, str):
        return spec
    return " + ".join(f"{w:g}*{n}" for n, w in spec.items())


def _exists(spec):
    names = [spec] if isinstance(spec, str) else list(spec)
    return all(_folds_avail(n) for n in names)


def infer_cv(spec):
    """spec = name or {name: weight}; -> (test_probs, val_probs, val_trues), CV-ensembled.

    Blends model fold-ensembles by weight on both the fixed test and the pooled CV val
    (fold order is identical across models, so the pooled-val arrays align)."""
    items = {spec: 1.0} if isinstance(spec, str) else spec
    wsum = sum(items.values())
    tp = vp = vy = None
    for name, w in items.items():
        cm = cv_model(name); f = w / wsum
        tp = f * cm["test"] if tp is None else tp + f * cm["test"]
        vp = f * cm["vprobs"] if vp is None else vp + f * cm["vprobs"]
        vy = cm["vtrues"]
    return tp, vp, vy


assert _exists(MODEL), f"{MODEL} not fully trained yet"
MODEL_NAME = spec_name(MODEL)

Y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in te_days])   # observed test labels
probs, vp, vy = infer_cv(MODEL)
pr, rc, thr = precision_recall_curve(vy[:, land].ravel(), vp[:, land].ravel())
f1c = 2 * pr * rc / (pr + rc + 1e-9)
THR = float(thr[np.argmax(f1c[:-1])])
M = (probs >= THR).astype(np.float32)                   # binarized prediction (test)
ap = average_precision_score(Y[:, land].ravel(), probs[:, land].ravel())
print(f"model: {MODEL_NAME}  ({N_FOLDS}-fold ensemble)")
print(f"test AUPRC {ap:.4f}  |  pooled-val-F1 threshold {THR:.3f}  |  "
      f"flagged cells/day {M[:, land].sum() / len(te_days):.1f}")


## 2. NWS warnings -> 50 km grid

A cell is **warned** on CDT day D if an NWS Flash-Flood (FF) or Areal-Flood (FA) *warning*
polygon is active during that CDT day and intersects the cell. Warning issue/expire are UTC;
we shift by -5 h to CDT (matching the label construction) and rasterize onto the same grid,
for the test days only.

In [ ]:
CDT = pd.Timedelta(hours=5)
te_ts = pd.to_datetime(te_days, format="%Y%m%d")
te_set = set(te_ts)

u = gpd.read_parquet(UNIFIED_PARQUET)
w = u[u["source"].isin(["ff_warning", "fa_warning"])].copy()
w["issue_cdt"] = w["issue_date"] - CDT
w["expire_cdt"] = w["expire_date"] - CDT
# keep warnings that could touch a test CDT day (cheap pre-filter)
w = w[(w["expire_cdt"] >= te_ts.min()) & (w["issue_cdt"] <= te_ts.max() + pd.Timedelta(days=1))]


def cdt_days(issue, expire):
    d = pd.date_range(issue.normalize(), expire.normalize(), freq="D")
    return d if len(d) <= 5 else d[:1]


w["day"] = [cdt_days(i, e) for i, e in zip(w["issue_cdt"], w["expire_cdt"])]
ev = w.explode("day", ignore_index=True)
ev = ev[ev["day"].isin(te_set)]

j = gpd.sjoin(cells_gdf, ev[["day", "geometry"]], predicate="intersects")
warn_map = {}
for day, g in j.groupby("day"):
    a = np.zeros((GRID_R, GRID_C), np.float32)
    a[g["R"], g["C"]] = 1.0
    warn_map[day.strftime("%Y%m%d")] = a

W = np.stack([warn_map.get(d, np.zeros((GRID_R, GRID_C), np.float32)) for d in te_days])
print(f"warnings rasterized: {len(ev):,} warning-days -> "
      f"{sum(v.sum() > 0 for v in warn_map.values())}/{len(te_days)} test days have >=1 warning")
print(f"warned land cells/day: {W[:, land].sum() / len(te_days):.1f}  "
      f"(observed floods/day: {Y[:, land].sum() / len(te_days):.1f})")

## 3. Metrics - model & warnings vs observed floods

Both predictions scored against the observed labels on land cells: precision, recall, F1, CSI
(exact and 1-grid-tolerant). The model additionally has AUPRC (probabilistic); warnings are
binary. `xbase` = AUPRC / base rate.

In [ ]:
def _dilate(m):
    o = m.copy()
    o[:-1, :] |= m[1:, :]; o[1:, :] |= m[:-1, :]
    o[:, :-1] |= m[:, 1:]; o[:, 1:] |= m[:, :-1]
    o[:-1, :-1] |= m[1:, 1:]; o[1:, 1:] |= m[:-1, :-1]
    o[:-1, 1:] |= m[1:, :-1]; o[1:, :-1] |= m[:-1, 1:]
    return o


def binary_metrics(pred, y):
    """P/R/F1/CSI exact + 1-grid for a binary prediction (N,R,C) vs labels."""
    pb = pred[:, land].ravel().astype(int)
    t = y[:, land].ravel().astype(int)
    tp = int((pb * t).sum()); fp = int((pb * (1 - t)).sum()); fn = int(((1 - pb) * t).sum())
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9); csi = tp / (tp + fn + fp + 1e-9)
    h = m1 = fa = 0
    for i in range(len(pred)):
        yt = (y[i] > 0.5) & land
        yp = (pred[i] > 0.5) & land
        h += int((yt & (_dilate(yp) & land)).sum())
        m1 += int((yt & ~(_dilate(yp) & land)).sum())
        fa += int((yp & ~(_dilate(yt) & land)).sum())
    p1 = h / (h + fa + 1e-9); r1 = h / (h + m1 + 1e-9)
    return dict(P=prec, R=rec, F1=f1, CSI=csi,
                F1_1=2 * p1 * r1 / (p1 + r1 + 1e-9), CSI1=h / (h + m1 + fa + 1e-9))


base = base_rate(te_days)
rows = []
mm = binary_metrics(M, Y)
mm["AUPRC"] = average_precision_score(Y[:, land].ravel(), probs[:, land].ravel())
mm["xbase"] = mm["AUPRC"] / base
rows.append(dict(predictor=f"{MODEL_NAME} (D-1 GOES)", **mm))
wm = binary_metrics(W, Y)
wm["AUPRC"] = float("nan"); wm["xbase"] = float("nan")
rows.append(dict(predictor="NWS warnings", **wm))

tbl = pd.DataFrame(rows).set_index("predictor")[
    ["AUPRC", "xbase", "P", "R", "F1", "CSI", "F1_1", "CSI1"]]
print(f"test base rate {base:.4f}  ({len(te_days)} days)\n")
print(tbl.round(3).to_string())

## 3b. Single models vs ensembles

Evaluate each single model (as its own K-fold ensemble) and the candidate cross-model
ensembles against observed floods on the fixed test set - each thresholded on its pooled CV
val. Shows whether blending beats the best single model.


In [ ]:
CANDIDATES = ["convgru_attn", "resnet3d", "cnn_attn", "xgb",
              {"resnet3d": 0.5, "convgru_attn": 0.5},
              {"cnn_attn": 0.5, "convgru_attn": 0.5},
              {"resnet3d": 0.4, "convgru_attn": 0.4, "xgb": 0.2}]

rows = []
for spec in CANDIDATES:
    if not _exists(spec):
        continue
    pp, vpp, vyy = infer_cv(spec)
    prc, rcc, thc = precision_recall_curve(vyy[:, land].ravel(), vpp[:, land].ravel())
    fc = 2 * prc * rcc / (prc + rcc + 1e-9)
    th = float(thc[np.argmax(fc[:-1])])
    apc = average_precision_score(Y[:, land].ravel(), pp[:, land].ravel())
    bm = binary_metrics((pp >= th).astype(np.float32), Y)
    rows.append(dict(model=spec_name(spec), AUPRC=apc, xbase=apc / base,
                     F1=bm["F1"], CSI=bm["CSI"], F1_1=bm["F1_1"], CSI1=bm["CSI1"]))

cand = pd.DataFrame(rows).set_index("model").sort_values("AUPRC", ascending=False)
print("fixed test - single models vs ensembles (sorted by AUPRC):\n")
print(cand.round(3).to_string())
_bst = cand.index[0]
print(f"\n>>> best by AUPRC: {_bst}  ({cand.iloc[0]['AUPRC']:.4f}, {cand.iloc[0]['xbase']:.1f}x)")


## 4. Who catches the observed floods?

Of every observed flood cell-day on the test set, how many are flagged by the **model only**,
**NWS only**, **both**, or **neither** - and the same for false alarms (flagged but no observed
flood). This shows whether the model adds coverage beyond the operational warnings.

In [ ]:
ml = M[:, land].astype(bool).ravel()
wl = W[:, land].astype(bool).ravel()
yl = Y[:, land].astype(bool).ravel()

pos = yl.sum()
both = int((ml & wl & yl).sum())
mo = int((ml & ~wl & yl).sum())
wo = int((~ml & wl & yl).sum())
neither = int((~ml & ~wl & yl).sum())
print(f"observed flood cell-days on test: {pos}")
print(f"  caught by BOTH        : {both:6d}  ({both / pos:5.1%})")
print(f"  caught by MODEL only  : {mo:6d}  ({mo / pos:5.1%})")
print(f"  caught by NWS only    : {wo:6d}  ({wo / pos:5.1%})")
print(f"  MISSED by both        : {neither:6d}  ({neither / pos:5.1%})")
print(f"\nrecall  model {(both + mo) / pos:.1%}   NWS {(both + wo) / pos:.1%}")
inter = int((ml & wl).sum()); union = int((ml | wl).sum())
print(f"model-vs-NWS footprint IoU: {inter / (union + 1e-9):.3f}  "
      f"(model flags {int(ml.sum())}, NWS flags {int(wl.sum())} cell-days)")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.barh(["both", "model only", "NWS only", "missed by both"],
        [both, mo, wo, neither], color=["#4c956c", "#e09f3e", "#6a4c93", "#c1121f"])
ax.set_xlabel("observed flood cell-days")
ax.set_title(f"Coverage of observed floods - {MODEL_NAME} vs NWS warnings (test)")
for i, v in enumerate([both, mo, wo, neither]):
    ax.text(v, i, f" {v / pos:.0%}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 5. Per-day maps

Random test days: observed floods, the model's binary prediction (@ val threshold), and the
NWS warning footprint. Change `SEED`/`N_SHOW`.

In [ ]:
SEED = 44
N_SHOW = 4


def _draw(ax, values, cmap, title=None, ylabel=None):
    gdf = cells_albers.copy(); gdf["v"] = values[RR, CC]
    gdf.boundary.plot(ax=ax, color="white", lw=0.1, zorder=2)
    gdf.plot(column="v", cmap=cmap, ax=ax, zorder=1, edgecolor="none", vmin=0, vmax=1)
    states.boundary.plot(ax=ax, color="0.5", lw=0.5, zorder=3)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, fontsize=11, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")


rng = np.random.default_rng(SEED)
sel = sorted(rng.choice(len(te_days), size=min(N_SHOW, len(te_days)), replace=False))
rows = [("Observed", Y, "Greens"), (f"{MODEL_NAME}", M, "Oranges"), ("NWS warnings", W, "Purples")]
fig, axes = plt.subplots(len(rows), len(sel), figsize=(3.4 * len(sel), 3.1 * len(rows)),
                         squeeze=False)
for c, i in enumerate(sel):
    ny = int((Y[i][land] > 0).sum())
    for r, (label, arr, cmap) in enumerate(rows):
        _draw(axes[r, c], arr[i], cmap,
              title=f"{te_days[i]}  ({ny} floods)" if r == 0 else None,
              ylabel=label if c == 0 else None)
fig.suptitle(f"Test days (seed {SEED}) - observed vs {MODEL_NAME} vs NWS warnings",
             fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout(); plt.show()

## 6. Aggregate spatial view

Summed over all test days: where floods were observed, where the model fired, and where NWS
warned - so persistent hits / gaps stand out geographically.

In [ ]:
agg = [("Observed flood-days", Y.sum(0)), (f"{MODEL_NAME} flagged-days", M.sum(0)),
       ("NWS warned-days", W.sum(0))]
mx = max(float(a[land].max()) for _, a in agg) or 1.0
fig, axes = plt.subplots(1, 3, figsize=(4.4 * 3, 4.2))
for ax, (title, a) in zip(axes, agg):
    gdf = cells_albers.copy(); gdf["v"] = np.where(land, a, np.nan)[RR, CC]
    gdf.plot(column="v", cmap="magma_r", ax=ax, vmin=0, vmax=mx, edgecolor="none")
    states.boundary.plot(ax=ax, color="0.5", lw=0.5)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
fig.colorbar(plt.cm.ScalarMappable(cmap="magma_r",
             norm=plt.Normalize(0, mx)), ax=axes, fraction=0.02, pad=0.01, label="days")
fig.suptitle("Aggregate over test days (0-{:.0f})".format(mx), fontsize=13, fontweight="bold")
plt.show()

## 7. Metric bar chart

Side-by-side precision / recall / F1 / CSI (exact + 1-grid) for the model vs NWS warnings,
both against observed floods.

In [ ]:
keys = ["P", "R", "F1", "CSI", "F1_1", "CSI1"]
labels = ["Prec", "Recall", "F1", "CSI", "F1@1", "CSI@1"]
xm = np.arange(len(keys))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(xm - 0.2, [mm[k] for k in keys], 0.4, label=f"{MODEL_NAME} (D-1 GOES)", color="#e09f3e")
ax.bar(xm + 0.2, [wm[k] for k in keys], 0.4, label="NWS warnings", color="#6a4c93")
ax.set_xticks(xm); ax.set_xticklabels(labels)
ax.set_ylabel("score (vs observed floods)")
ax.set_title(f"{MODEL_NAME} vs NWS warnings on the test set")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 7b. Proper comparison — performance diagram + matched operating points

AUPRC can't score NWS (it's binary), and comparing the model's best-F1 point to NWS is unfair
because they fire at very different alarm rates. The **performance diagram** (Roebber 2009) is
the standard: **POD (recall)** vs **Success Ratio (1−FAR = precision)**, with **CSI** as curved
contours and **frequency bias** as diagonals. The model is a curve (swept threshold); NWS is a
single point. If the model's curve passes above-and-right of the NWS point it dominates (higher
CSI). Below the diagram we also read off the **matched operating points**: the model's precision
*at NWS's recall*, and its recall *at NWS's precision* — a like-for-like head-to-head.

In [ ]:
from sklearn.metrics import precision_recall_curve


def compare_vs_nws(probs, W, Y, ttl):
    """Performance diagram + matched-operating-point scalars, over all land cell-days.

    Model = probabilistic (threshold-swept curve); NWS = one binary point. CSI contours +
    frequency-bias diagonals. Prints POD/SR/CSI/bias for NWS, the model at its val threshold,
    and the model re-thresholded to match NWS's recall / precision."""
    t = Y[:, land].ravel().astype(bool)
    p = probs[:, land].ravel()
    wl = (W[:, land] > 0.5).ravel()

    def stats(mask):
        tp = int((mask & t).sum()); fp = int((mask & ~t).sum()); fn = int((~mask & t).sum())
        pod = tp / (tp + fn + 1e-9); sr = tp / (tp + fp + 1e-9)
        csi = tp / (tp + fp + fn + 1e-9); bias = (tp + fp) / (tp + fn + 1e-9)
        return pod, sr, csi, bias

    def csi_of(pod, sr):
        return 1.0 / (1 / sr + 1 / pod - 1) if (pod > 0 and sr > 0) else 0.0

    pod_n, sr_n, csi_n, bias_n = stats(wl)                 # NWS operating point
    pod_m, sr_m, csi_m, bias_m = stats(p >= THR)           # model @ its val-F1 threshold
    pr, rc, _ = precision_recall_curve(t, p)               # model frontier (SR=pr, POD=rc)
    prec_at_nws_recall = float(pr[rc >= pod_n].max()) if (rc >= pod_n).any() else float("nan")
    rec_at_nws_prec = float(rc[pr >= sr_n].max()) if (pr >= sr_n).any() else 0.0

    # ---- performance diagram ----
    fig, ax = plt.subplots(figsize=(6.2, 6))
    g = np.linspace(1e-3, 1, 250); SR, POD = np.meshgrid(g, g)
    CSI = 1.0 / (1 / SR + 1 / POD - 1)
    ax.contourf(SR, POD, CSI, levels=np.linspace(0, 1, 11), cmap="Blues", alpha=0.30)
    cl = ax.contour(SR, POD, CSI, levels=[.05, .1, .15, .2, .3, .4, .6, .8],
                    colors="0.5", linewidths=0.6)
    ax.clabel(cl, fmt="%.2f", fontsize=7)
    for b in [0.5, 1, 1.5, 2, 3, 5]:                       # frequency-bias diagonals POD=b*SR
        ax.plot(g, np.clip(b * g, 0, 1), "--", color="0.75", lw=0.6)
        xb = min(1.0, 1.0 / b)
        ax.text(xb, min(1.0, b * xb) + 0.01, f"{b:g}", color="0.55", fontsize=7, ha="center")
    ax.plot(pr, rc, "-", color="#118ab2", lw=2, label="model (sweep threshold)")
    ax.plot(sr_m, pod_m, "o", color="#118ab2", ms=9,
            label=f"model @ val-F1 thr (CSI {csi_m:.3f})")
    ax.plot(sr_n, pod_n, "s", color="#6a4c93", ms=10,
            label=f"NWS warnings (CSI {csi_n:.3f})")
    ax.set(xlim=(0, 1), ylim=(0, 1), aspect="equal",
           xlabel="Success Ratio  (1 − FAR = precision)", ylabel="POD  (recall)", title=ttl)
    ax.legend(loc="upper right", fontsize=8, framealpha=0.92)
    plt.tight_layout(); plt.show()

    # ---- matched-operating-point table ----
    rows = [
        ("NWS warnings",            pod_n, sr_n, csi_n, bias_n),
        ("model @ val-F1 thr",      pod_m, sr_m, csi_m, bias_m),
        ("model @ NWS's recall",    pod_n, prec_at_nws_recall,
         csi_of(pod_n, prec_at_nws_recall), float("nan")),
        ("model @ NWS's precision", rec_at_nws_prec, sr_n,
         csi_of(rec_at_nws_prec, sr_n), float("nan")),
    ]
    print(f"{ttl}\n{'predictor':<26}{'POD':>7}{'SR':>7}{'CSI':>7}{'bias':>7}")
    for nm, pod, sr, csi, bias in rows:
        bs = f"{bias:>7.2f}" if bias == bias else f"{'-':>7}"
        print(f"{nm:<26}{pod:>7.3f}{sr:>7.3f}{csi:>7.3f}{bs}")
    print(f"\n  at NWS's recall ({pod_n:.1%}):    model precision {prec_at_nws_recall:.1%}"
          f"  vs NWS {sr_n:.1%}")
    print(f"  at NWS's precision ({sr_n:.1%}): model recall    {rec_at_nws_prec:.1%}"
          f"  vs NWS {pod_n:.1%}")
    better = "model" if csi_of(pod_n, prec_at_nws_recall) > csi_n else "NWS warnings"
    print(f"  -> at matched detection (equal recall), higher CSI: {better}")


compare_vs_nws(probs, W, Y, "Fixed test (2019-2025 held-out) — model vs NWS warnings")

## 8. 2026 held-out test (all cached days — NOT in training or CV)

A bonus most-recent test: all **65 cached 2026 days** (Jan 1 – Mar 31). 2026 is excluded from
training and from the CV folds (`foldsplit` drops `split == "unused"`), so this is a clean
held-out sample. The model prediction is the same **K-fold ensemble**, thresholded at the same
**pooled-CV-val** operating point (`THR`) — no refitting on 2026.

**Caveat:** groundsource labels end **2026-02-03**; from Feb 10 on the union label is
effectively **storm-only**, so floods there are undercounted and *both* the model and NWS look
pessimistic on those days. Read the 2026 numbers with that in mind.

In [ ]:
# ---- 2026 held-out test: all cached 2026 days (never trained on / in CV) ----
man26 = pd.read_parquet(CACHE_DIR / "manifest.parquet")
man26["label_day"] = pd.to_datetime(man26["label_day"])
days26 = [d.strftime("%Y%m%d") for d in man26.loc[man26.label_day.dt.year == 2026, "label_day"]]
days26 = [d for d in days26 if (CACHE_DIR / f"{d}_sum.npy").exists()]
te26_ts = pd.to_datetime(days26, format="%Y%m%d")
Y26 = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days26])


def cv_ensemble_probs(spec, days):
    """6-fold ensemble probability for a spec (name or {name: w}) on arbitrary days."""
    items = {spec: 1.0} if isinstance(spec, str) else spec
    wsum = sum(items.values())
    out = None
    for name, w in items.items():
        folds = _folds_avail(name)
        fp = None
        for k in folds:
            pr, _ = _infer_ckpt(name, _fold_ckpt(name, k), days)
            fp = pr if fp is None else fp + pr
        fp = fp / max(1, len(folds))
        out = (w / wsum) * fp if out is None else out + (w / wsum) * fp
    return out


probs26 = cv_ensemble_probs(MODEL, days26)
M26 = (probs26 >= THR).astype(np.float32)          # SAME operating threshold (pooled CV val)

# NWS warnings on the 2026 CDT days (reuse the unified layer `u` + cell-6 `cdt_days`/`CDT`)
w26 = u[u["source"].isin(["ff_warning", "fa_warning"])].copy()
w26["issue_cdt"] = w26["issue_date"] - CDT
w26["expire_cdt"] = w26["expire_date"] - CDT
te26_set = set(te26_ts)
w26 = w26[(w26["expire_cdt"] >= te26_ts.min()) &
          (w26["issue_cdt"] <= te26_ts.max() + pd.Timedelta(days=1))]
w26["day"] = [cdt_days(i, e) for i, e in zip(w26["issue_cdt"], w26["expire_cdt"])]
ev26 = w26.explode("day", ignore_index=True)
ev26 = ev26[ev26["day"].isin(te26_set)]
j26 = gpd.sjoin(cells_gdf, ev26[["day", "geometry"]], predicate="intersects")
wmap26 = {}
for day, g in j26.groupby("day"):
    a = np.zeros((GRID_R, GRID_C), np.float32); a[g["R"], g["C"]] = 1.0
    wmap26[day.strftime("%Y%m%d")] = a
W26 = np.stack([wmap26.get(d, np.zeros((GRID_R, GRID_C), np.float32)) for d in days26])

base26 = float(Y26[:, land].mean())
mm26 = binary_metrics(M26, Y26)
mm26["AUPRC"] = average_precision_score(Y26[:, land].ravel(), probs26[:, land].ravel())
mm26["xbase"] = mm26["AUPRC"] / base26
wm26 = binary_metrics(W26, Y26)
wm26["AUPRC"] = float("nan"); wm26["xbase"] = float("nan")
tbl26 = pd.DataFrame([
    dict(predictor=f"{MODEL_NAME} (D-1 GOES)", **mm26),
    dict(predictor="NWS warnings", **wm26),
]).set_index("predictor")[["AUPRC", "xbase", "P", "R", "F1", "CSI", "F1_1", "CSI1"]]

print(f"2026 held-out test: {len(days26)} days ({days26[0]}-{days26[-1]})  base rate {base26:.4f}")
print("  NOTE: groundsource truncated after 2026-02-03 -> Feb 10+ are storm-only labels\n")
print(tbl26.round(3).to_string())

ml, wl, yl = (M26[:, land].astype(bool).ravel(), W26[:, land].astype(bool).ravel(),
              Y26[:, land].astype(bool).ravel())
pos = int(yl.sum())
both = int((ml & wl & yl).sum()); mo = int((ml & ~wl & yl).sum())
wo = int((~ml & wl & yl).sum()); nn = int((~ml & ~wl & yl).sum())
print(f"\nobserved 2026 flood cell-days: {pos}")
print(f"  both {both} ({both/pos:.0%}) | model-only {mo} ({mo/pos:.0%}) | "
      f"NWS-only {wo} ({wo/pos:.0%}) | missed {nn} ({nn/pos:.0%})")
print(f"recall  model {(both+mo)/pos:.1%}   NWS {(both+wo)/pos:.1%}  "
      f"(flagged/day: model {M26[:,land].sum()/len(days26):.1f}, NWS {W26[:,land].sum()/len(days26):.1f})")

### 8b. 2026 per-day maps

Random 2026 held-out days: observed floods, the model's binary prediction (@ the same
pooled-CV-val threshold), and the NWS warning footprint. Change `SEED26`/`N_SHOW26`.

In [ ]:
SEED26 = 88          # <- change to resample which 2026 days are shown
N_SHOW26 = 4

rng26 = np.random.default_rng(SEED26)
sel26 = sorted(rng26.choice(len(days26), size=min(N_SHOW26, len(days26)), replace=False))
rows26 = [("Observed", Y26, "Greens"), (MODEL_NAME, M26, "Oranges"),
          ("NWS warnings", W26, "Purples")]

fig, axes = plt.subplots(len(rows26), len(sel26),
                         figsize=(3.4 * len(sel26), 3.1 * len(rows26)), squeeze=False)
for c, i in enumerate(sel26):
    ny = int((Y26[i][land] > 0).sum())
    for r, (label, arr, cmap) in enumerate(rows26):
        _draw(axes[r, c], arr[i], cmap,
              title=f"{days26[i]}  ({ny} floods)" if r == 0 else None,
              ylabel=label if c == 0 else None)
fig.suptitle(f"2026 held-out days (seed {SEED26}) - observed vs {MODEL_NAME} vs NWS warnings",
             fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout(); plt.show()

### 8c. 2026 — performance diagram vs NWS\n\nThe same proper comparison on the 2026 held-out days (base rate is low / storm-only after Feb 3, so read POD/SR/CSI more than absolutes).

In [ ]:
compare_vs_nws(probs26, W26, Y26, "2026 held-out — model vs NWS warnings")